In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_style("whitegrid")

In [3]:
DATA_PATH = "E:\Data Analyst Projects\project shopping trend.csv"  # <-- update to your file location
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.head())
print(df.info())

Shape: (3900, 19)
   Customer_ID  Age Gender Item_Purchased  Category  Purchase_Amount_USD  \
0            1   55   Male         Blouse  Clothing                   53   
1            2   19   Male        Sweater  Clothing                   64   
2            3   50   Male          Jeans  Clothing                   73   
3            4   21   Male        Sandals  Footwear                   90   
4            5   45   Male         Blouse  Clothing                   49   

        Location Size      Color  Season  Review_Rating Subscription_Status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    L     Maroon  Winter            3.1                 Yes   
2  Massachusetts    S     Maroon  Spring            3.1                 Yes   
3   Rhode Island    M     Maroon  Spring            3.5                 Yes   
4         Oregon    M  Turquoise  Spring            2.7                 Yes   

  Payment_Method  Shipping_Type Discount_Applied P

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\DELL\AppData\Local\Temp\ipykernel_7920\1409548706.py:1: SyntaxWarning: invalid escape sequence '\D'
  DATA_PATH = "E:\Data Analyst Projects\project shopping trend.csv"  # <-- update to your file location


In [7]:
df.columns = df.columns.str.strip().str.replace(" ", "_")

# Drop exact duplicates and rows with no purchase amount
df = df.drop_duplicates()
df = df.dropna(subset=["Purchase_Amount_USD"])

# Standardize categorical text fields
for col in ["Gender", "Category", "Season", "Payment_Method",
            "Subscription_Status", "Discount_Applied", "Promo_Code_Used"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

print("\nMissing values after cleaning:\n", df.isna().sum())



Missing values after cleaning:
 Customer_ID                 0
Age                         0
Gender                      0
Item_Purchased              0
Category                    0
Purchase_Amount_USD         0
Location                    0
Size                        0
Color                       0
Season                      0
Review_Rating               0
Subscription_Status         0
Payment_Method              0
Shipping_Type               0
Discount_Applied            0
Promo_Code_Used             0
Previous_Purchases          0
Preferred_Payment_Method    0
Frequency_of_Purchases      0
dtype: int64


In [15]:
# Categories with high sales volume but lower discount usage are good
# candidates for *increased* targeted discounting to boost conversion.
# Categories with already-high discount usage and high revenue (e.g.
# Clothing, Accessories) show discounts are effective volume drivers.


discount_by_category = (
    df.groupby("Category")["Discount_Applied"]
      .apply(lambda x: (x == "Yes").mean() * 100)
      .sort_values(ascending=False)
)
print("\n% of orders with discount applied, by category:\n", discount_by_category)

revenue_by_category = df.groupby("Category")["Purchase_Amount_USD"].sum().sort_values(ascending=False)
print("\nTotal revenue by category:\n", revenue_by_category)


% of orders with discount applied, by category:
 Category
Outerwear      44.444444
Accessories    43.790323
Footwear       43.238731
Clothing       42.084053
Name: Discount_Applied, dtype: float64

Total revenue by category:
 Category
Clothing       104264
Accessories     74200
Footwear        36093
Outerwear       18524
Name: Purchase_Amount_USD, dtype: int64


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
revenue_by_category.plot(kind="bar", ax=axes[0], color="#4C72B0", title="Revenue by Category")
discount_by_category.plot(kind="bar", ax=axes[1], color="#DD8452", title="% Orders Discounted by Category")
plt.tight_layout()
plt.savefig("category_discount_analysis.png", dpi=150)
plt.close()

In [13]:
# 4. CARD SPENDING BY AGE / SEASON / LOCATION
# ----------------------------------------------------------------------
card_df = df[df["Payment_Method"].str.contains("Card", case=False, na=False)]

spend_by_age = card_df.groupby("Age")["Purchase_Amount_USD"].mean()
spend_by_season = card_df.groupby("Season")["Purchase_Amount_USD"].sum().sort_values(ascending=False)
spend_by_location = card_df.groupby("Location")["Purchase_Amount_USD"].sum().sort_values(ascending=False)

print("\nAvg card spend by age (head):\n", spend_by_age.head())
print("\nCard spend by season:\n", spend_by_season)
print("\nTop 10 locations by card spend:\n", spend_by_location.head(10))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
spend_by_age.plot(ax=axes[0], title="Avg Card Spend by Age")
spend_by_season.plot(kind="bar", ax=axes[1], title="Card Spend by Season", color="#55A868")
spend_by_location.head(10).plot(kind="bar", ax=axes[2], title="Top 10 Locations (Card Spend)", color="#C44E52")
plt.tight_layout()
plt.savefig("card_spending_patterns.png", dpi=150)
plt.close()


Avg card spend by age (head):
 Age
18    57.181818
19    56.766667
20    61.368421
21    71.230769
22    68.894737
Name: Purchase_Amount_USD, dtype: float64

Card spend by season:
 Season
Fall      21694
Winter    20885
Spring    19099
Summer    18007
Name: Purchase_Amount_USD, dtype: int64

Top 10 locations by card spend:
 Location
Montana           2445
Illinois          2443
North Dakota      2290
Tennessee         2118
Utah              2036
North Carolina    2026
Maryland          1872
Arkansas          1853
Kentucky          1827
Nevada            1821
Name: Purchase_Amount_USD, dtype: int64


In [19]:
# CUSTOMER CLUSTERING
# ----------------------------------------------------------------------
cluster_df = df.copy()

le_gender = LabelEncoder()
le_season = LabelEncoder()
le_location = LabelEncoder()

cluster_df["Gender_enc"] = le_gender.fit_transform(cluster_df["Gender"])
cluster_df["Season_enc"] = le_season.fit_transform(cluster_df["Season"])
cluster_df["Location_enc"] = le_location.fit_transform(cluster_df["Location"])

features = [
    "Age", "Purchase_Amount_USD", "Previous_Purchases",
    "Gender_enc", "Season_enc", "Location_enc"
]
X = cluster_df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow method to choose K
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(K_range), inertias, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")
plt.savefig("elbow_method.png", dpi=150)
plt.close()

In [21]:
# Final clustering with K=3 (High / Medium / Low value segments)
K_FINAL = 3
kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
cluster_df["Cluster"] = kmeans.fit_predict(X_scaled)

In [23]:
# Label clusters by average purchase amount: High / Medium / Low value
cluster_avg_spend = cluster_df.groupby("Cluster")["Purchase_Amount_USD"].mean().sort_values(ascending=False)
rank_to_label = {cluster: label for cluster, label in
                  zip(cluster_avg_spend.index, ["High Value", "Medium Value", "Low Value"])}
cluster_df["Segment"] = cluster_df["Cluster"].map(rank_to_label)

print("\nCluster sizes:\n", cluster_df["Segment"].value_counts())
print("\nAvg purchase amount per segment:\n", cluster_df.groupby("Segment")["Purchase_Amount_USD"].mean())
print("\nRevenue contribution per segment:\n", cluster_df.groupby("Segment")["Purchase_Amount_USD"].sum())



Cluster sizes:
 Segment
Medium Value    1330
Low Value       1322
High Value      1248
Name: count, dtype: int64

Avg purchase amount per segment:
 Segment
High Value      60.249199
Low Value       58.974281
Medium Value    60.094737
Name: Purchase_Amount_USD, dtype: float64

Revenue contribution per segment:
 Segment
High Value      75191
Low Value       77964
Medium Value    79926
Name: Purchase_Amount_USD, dtype: int64


In [25]:
# Dominant factors per cluster (mean of each scaled feature)
cluster_profile = cluster_df.groupby("Segment")[features].mean()
print("\nCluster feature profile:\n", cluster_profile)


Cluster feature profile:
                     Age  Purchase_Amount_USD  Previous_Purchases  Gender_enc  \
Segment                                                                        
High Value    44.007212            60.249199           24.596154         0.0   
Low Value     43.629349            58.974281           26.032526         1.0   
Medium Value  44.562406            60.094737           25.383459         1.0   

              Season_enc  Location_enc  
Segment                                 
High Value      1.466346     24.250801  
Low Value       2.503026     24.723147  
Medium Value    0.513534     23.827068  


In [27]:
# Visualize clusters via PCA (2D projection)
pca = PCA(n_components=2)
pca_components = pca.fit_transform(X_scaled)
cluster_df["PC1"] = pca_components[:, 0]
cluster_df["PC2"] = pca_components[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=cluster_df, x="PC1", y="PC2", hue="Segment", palette="viridis", alpha=0.7)
plt.title("Customer Segments (PCA Projection)")
plt.savefig("customer_segments_pca.png", dpi=150)
plt.close()

In [29]:
# 6. TARGETED MARKETING RECOMMENDATIONS (based on segment profile)
# ----------------------------------------------------------------------
recommendations = {
    "High Value": "Loyalty rewards, early access to new collections, premium bundle offers.",
    "Medium Value": "Seasonal discount campaigns, cross-sell bundles to lift average order value.",
    "Low Value": "Re-engagement promo codes, first-purchase discounts, low-cost entry bundles."
}
for seg, rec in recommendations.items():
    print(f"\n{seg} Segment -> {rec}")

cluster_df.to_csv("customer_segments_output.csv", index=False)
print("\nSaved cluster output to customer_segments_output.csv")



High Value Segment -> Loyalty rewards, early access to new collections, premium bundle offers.

Medium Value Segment -> Seasonal discount campaigns, cross-sell bundles to lift average order value.

Low Value Segment -> Re-engagement promo codes, first-purchase discounts, low-cost entry bundles.

Saved cluster output to customer_segments_output.csv
